# Recent Feature Additions

## Version 2.3.2

### MCP server enhancements

The MCP integration includes expanded tooling and better runtime control:

- Added `reduce_system` and `save_system` operations for model reduction and export workflows.
- Added runtime tool gating with `set_tool_calls_enabled` and `get_tool_calls_enabled`.
- Added `model_ref` interoperability for path-based tools (direct path fields or registry-backed lookup).

### Tariff model update

Time-of-use tariff season tagging uses month-based tagging:

- `SeasonalTOURates.season` now uses `Month` enum values.
- Examples now use explicit month values (for example, `Month.JULY`, `Month.JANUARY`).

### Minor bug fixes

This patch release also includes targeted stability fixes:

- **Issue #163**: Adjusted bus/branch phase validation to avoid false errors in multigraph cases (for example, parallel single-phase branches between three-phase buses) and neutral-handling edge cases.
- **Issue #112**: Reduced repeated reducer log spam by avoiding repeated graph/warning emissions during reduction workflows.
- **Issue #114**: Standardized naming from `timeseries` to `time_series` across APIs and docs for consistency with infrasys conventions.
- **Issue #157**: Fixed capacitor aggregation edge cases to prevent division-by-zero when all resistance or reactance inputs are zero.

### Documentation updates

MCP docs and release notes were refreshed to reflect the current tooling and bug-fix behavior.

## Version 2.3.0

### Introduction of MCP (Model Context Protocol) server

This release marks the introduction of a new Model Context Protocol (MCP) server for GDM, enabling AI agents and language models to interact with grid-data-models through standardized tools:

- **`gdm-mcp-server`**: A new CLI entry point that exposes GDM functionality as 20+ MCP tools.
- **Core tool categories**:
  - System diagnosis and fixing (analyze, diagnose, apply fixes, suggest improvements)
  - System operations (merge, split by feeder/substation, reduce complexity)
  - System inspection (query components, get relationships, analyze topology)
  - Time-series and subsystem management (export by buses, time series summary)
  - Documentation and knowledge access (search GDM docs, get code examples, API references)

### User impact

- GDM can now be integrated with AI tools and agents that support the MCP protocol.
- Claude Desktop and other MCP-compatible clients can connect to the server for enhanced grid analysis workflows.
- Useful for automated system diagnostics, batch model processing, and programmatic access via language models.

## Version 2.2.0

### Infrastructure and dependency migration

This release focused on migration and compatibility work required for the next feature cycle:

- Upgraded to `infrasys~=1.0` and aligned core distribution-system code paths with the new InfraSys interfaces.
- Added/updated upgrade-handler migration support from `2.1.5` to `2.2.0` (`migrate_component_metadata`) to preserve component counts while normalizing metadata.
- Applied dependency compatibility fixes for Pydantic and related packaging/versioning updates.
- Updated test matrix and release/deploy workflow pieces used by CI.
- Included docs/build fixes (including Jupyter Book deployment/link corrections).

### User impact

- Existing models can still be loaded through upgrade handlers when moving from 2.1.x to 2.2.x.
- Most changes are internal compatibility and release engineering updates rather than new end-user APIs.

## Version 2.1.6

### Release and packaging update

This release was a lightweight patch release focused on release automation:

- Updated deployment workflow configuration (`.github/workflows/deploy.yml`).

### User impact

- No major model/API behavior changes were introduced in this tag.
- Primarily improves packaging/release pipeline reliability.

## Version 2.2.1

### Deterministic directed graph construction

Recent fixes make the `DistributionSystem.get_directed_graph` behaviour deterministic:

- DFS traversal is now deterministic: neighbors and parallel edges are iterated in sorted order.
- Cycle pruning is deterministic: when producing a radial network the edge removed from each cycle is chosen by lexicographic edge-name ordering (with tie-breakers).

These changes remove flakiness from tests that assert exact node/edge counts or specific pruned edges.

```bash
pytest tests/test_distribution_graph.py::test_directed_graph_multiple_loops_pruning -q
```

If you rely on a specific choice of which parallel edge is used in analysis, prefer to inspect the undirected graph (`get_undirected_graph`) or explicitly set component names to control pruning selection.


###  Matrix impedance calculation fixes

#### Correct unit handling and length scaling

Fixed issues in `GeometryBranch.to_matrix_representation` and `convert_geometry_to_matrix_representation` where unit conversion and length scaling could produce incorrect impedance magnitudes. The conversion now consistently uses the system's unit definitions and converts lengths before computing per-unit matrices.

#### Kron reduction stability improvements

Improved numerical stability in `MatrixImpedanceBranch.kron_reduce()` to avoid introducing small imaginary/real artifacts during network reduction.

#### Notes for users

- If you previously converted geometry branches, re-run `convert_geometry_to_matrix_representation()` to pick up corrected impedance values.
- See `tests/test_impedence_calc.py` for unit tests covering these fixes.


## Version 2.1.5

### Graph bug fix

The graphs returned from GDM are type `MultiGraph` and `MultiDiGraph`. This allows user to represent multiple single-phase components as parallel edges in the graph.

```{note}
* Use of multiple parallel edges results in cycles in the graph. 
* A helper static method `get_cycles` has been added to the `DistributionSystem` class.
    * Unlike the networkX's 'simple_cycles' method, this function ignores these parallel edges connecting two nodes and only returns cycles with lengths greater than two edges.

```

### GDF Bug Fix

DistributionSystem now uses system CRS when exporting to GeoDataFrame format.

### Three-phase Model Reduction Algorithm Bug Fix

Bug fixes in the three-phase model reduction algorithm



In [2]:
from gdm.distribution.market import DistributionTariff

DistributionTariff.example().pprint()

DistributionTariff(
    name='Residential Summer Tariff',
    utility='Example Utility',
    customer_class=<CustomerClass.RESIDENTIAL: 'residential'>,
    fixed_charge=FixedCharge(name='', amount=15.0, description='Monthly fixed customer charge'),
    seasonal_tou=[
        SeasonalTOURates(
            name='',
            season=<Season.SUMMER: 'summer'>,
            tou_periods=[
                TOURatePeriod(
                    name='',
                    start_time=datetime.time(14, 0),
                    end_time=datetime.time(20, 0),
                    rate=0.25,
                    period_type=<TOUPeriodType.PEAK: 'peak'>
                ),
                TOURatePeriod(
                    name='',
                    start_time=datetime.time(20, 0),
                    end_time=datetime.time(23, 59),
                    rate=0.15,
                    period_type=<TOUPeriodType.OFF_PEAK: 'off_peak'>
                )
            ]
        ),
        SeasonalTOURates(
            name='',
            season=<Season.WINTER: 'winter'>,
            tou_periods=[
                TOURatePeriod(
                    name='',
                    start_time=datetime.time(14, 0),
                    end_time=datetime.time(20, 0),
                    rate=0.25,
                    period_type=<TOUPeriodType.PEAK: 'peak'>
                ),
                TOURatePeriod(
                    name='',
                    start_time=datetime.time(0, 0),
                    end_time=datetime.time(6, 0),
                    rate=0.1,
                    period_type=<TOUPeriodType.OFF_PEAK: 'off_peak'>
                )
            ]
        )
    ],
    demand_charges=[
        DemandCharge(
            name='',
            rate=12.5,
            billing_demand_basis=<BillingDemandBasis.PEAK_15MIN: 'peak_15min'>,
            time_applicability=[
                TOURatePeriod(
                    name='',
                    start_time=datetime.time(14, 0),
                    end_time=datetime.time(20, 0),
                    rate=0.25,
                    period_type=<TOUPeriodType.PEAK: 'peak'>
                )
            ]
        )
    ],
    tiered_energy_charges=[TieredRate(name='', upper_limit_kwh=500.0, rate=0.12)]
)

## Version 2.1.3
### Plotting bug fix

Plotting was broken after the last release. The bug has now been fixed.

### Tariff model added

The tariff model can now be imported using the code snippet below. In a future release, we will add model representations for aggregators and other market models.


## Version 2.1.2
### Support to line impedance calculations

$\quad$ `GeometryBranch` models can now be converted to `MatrixImpedanceBranch` model representation using the `to_matrix_representation` method.

```python
from gdm.distribution.components import GeometryBranch, MatrixImpedanceBranch

g = GeometryBranch.example()
h: MatrixImpedanceBranch = g.to_matrix_representation()
```

$\quad$ All `GeometryBranch` models in a given system can noe be replaced with `MatrixImpedanceBranch` model representation using the `convert_geometry_to_matrix_representation` method for DistributionSystem objects.

```python
from gdm.distribution import DistributionSystem

sys = DistributionSystem.from_json(filename=<path to model>) 
sys.convert_geometry_to_matrix_representation()
```


### Change to TrackChange object

$\quad$ `update_date` has been changed to `timestamp`. Now requires datetime object rather than date object..

```python
class TrackedChange(InfraSysBaseModel):
    .
    .
    .
    
    update_date: Annotated[
        timestamp | None,
        Field(
            None,
            description="If these changes are to be applied on specific timestamp, provide a timestamp, else leave it blank",
        ),
    ]
    .
    .
    .
```

## Version 2.1.0
### Improved validation for physical quantities

$\quad$ All Positive quantities have been removed. Additional constraints are now applied using Pydantic Field class. Reduces the number of class definations for physical quantities significantly.

### Initial implementation to manage backward compatability for DistributionSystems

$\quad$ Going forward (v2.0.0 onwards), users will be able to load older models with newer GDM installations by passing the upgrade handler object, when loading the model from a json file.

```python
from gdm.distribution import DistributionSystem
from gdm.distribution.upgrade_handler.upgrade_handler import UpgradeHandle

upgrade_handler = UpgradeHandler()
DistributionSystem.from_json(filename=<path to model>) 
upgrade_handler=upgrade_handler.upgrade)
```

### Support  to create a deepcopy of GDM system

$\quad$ Added functionality to easily create a deep copy of any GDM system object, ensuring that modifications to the copy do not affect the original instance.

```python
from gdm.distribution import DistributionSystem

system = DistributionSystem.from_json(filename=<path to model>)
system_copy = system.deepcopy()
```